In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
# from moc.models.mqf2.lightning_module import MQF2LightningModule
from moc.models.mixture.mixture_model2 import MixtureLightningModule
from moc.models.gaussian.gaussian import GaussianLightningModule
from moc.models.trainers.lightning_trainer import get_lightning_trainer
from moc.datamodules.real_datamodule import RealDataModule
import numpy as np
import matplotlib.pyplot as plt
from moc.metrics.distribution_metrics import pce
import torch
import wandb

In [3]:
config = get_config()
config.device = 'cuda'
torch.manual_seed(42)
data_group, data_name = 'mulan', 'oes10'
seed = 42

In [11]:
rc = RunConfig(config, data_group, data_name, seed=seed)
datamodule = RealDataModule(rc, seed=seed, num_workers=8)
p, q = datamodule.input_dim, datamodule.output_dim #268,16
prerank = 'density'
l = 1

In [12]:
model = GaussianLightningModule(p, q, lambda_reg = l, reg_type = 'pce-kde', prerank = prerank)
trainer = get_lightning_trainer(rc)
trainer.fit(model, datamodule)
wandb.finish()
model.to(config.device)
model.eval()

/home/ubuntu/anaconda3/envs/multicalibration/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/ubuntu/anaconda3/envs/multicalibration/lib/pyt ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/ubuntu/anaconda3/envs/multicalibration/lib/python3.12/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /mnt/default/elnura_workspace/Multivariate-recalibration/Multicalibration/Projected_PIT_calibration/logs/2025-06-18/10-26-35/mulan/oes10/None/0/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]


train/total_loss,██▇▇▇▆▆▅▅▄▃▃▂▁▁▁▂▂▂▂▂▂▂▂▁▁▁
val/total_loss,██▇▇▆▅▅▅▄▃▂▁▁▂▃▃▃▃▄▃▄▃▃▂▂▂▂
train/total_loss,0.04918
val/total_loss,0.10272


GaussianLightningModule(
  (model): MLP(
    (layers): ModuleList(
      (0): Linear(in_features=298, out_features=100, bias=True)
      (1): Linear(in_features=100, out_features=100, bias=True)
      (2): Linear(in_features=100, out_features=152, bias=True)
    )
  )
)

In [ ]:
torch.set_printoptions(precision=2, sci_mode=False, threshold=float('inf'), edgeitems=40, linewidth=200)
with torch.no_grad():
    for x, y in datamodule.test_dataloader():
        print(y.min(), y.max())
        x = x.to(config.device)
        y = y.to(config.device)
        dist = model.predict(x)
        # pce_values, cdf_values, extra = pce(dist, y, n_samples=100, prerank=prerank, setup='real', mode='test')
        # pces.append(pce_values)
        # cdfs.append(cdf_values)
        # if prerank == 'pca':
        #     explained_var = torch.from_numpy(extra).to(pce_values.device)
        #     weights.append(explained_var)

tensor(-0.80) tensor(8.23)
